In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import pathlib

pd.options.display.max_rows = 100
pd.options.display.max_seq_items = 100

db_files = list(pathlib.Path('../extraction/data').glob("iib-*.db"))
db_file = db_files.pop()

if not db_file:
    raise Exception("No DB files found")

print("Connecting to database " + db_file.name)
con = sqlite3.connect(db_file)

In [ ]:
measurements = pd.read_sql_query("""
    SELECT
        measurements.*,
        urls.category
    FROM
        measurements
        LEFT JOIN urls ON measurements.url = urls.url
    WHERE
        TYPE = 'mx-root'
""", con)
measurements

In [ ]:
pd.concat([
    measurements.measurement.value_counts(), 
    measurements.measurement.value_counts(normalize=True).mul(100)
], axis=1, keys=["Count", "%"])

In [ ]:
mx_freq_df = pd.concat(
    [
        measurements.groupby('category').measurement.value_counts(), 
        measurements.groupby('category').measurement.value_counts(normalize=True).mul(100)
    ]
    , axis=1, keys=["Count", "%"]
).sort_values("%", ascending=False).query('Count > 1').head(100)
mx_freq_df

In [ ]:
webhost_measurements = pd.read_sql_query("""
    SELECT
        measurements.*,
        urls.category
    FROM
        measurements
        LEFT JOIN urls ON measurements.url = urls.url
    WHERE
        TYPE = 'webhost'
""", con)
webhost_measurements

In [ ]:
webhost_freq_df = pd.concat(
    [
        webhost_measurements.groupby('category').measurement.value_counts(), 
        webhost_measurements.groupby('category').measurement.value_counts(normalize=True).mul(100)
    ]
    , axis=1, keys=["Count", "%"]
)
webhost_freq_df.sort_values("%", ascending=False).query('Count > 1').head(100)

In [ ]:
fig, ax = plt.subplots()
mx_freq_df["%"].sort_values().plot(ax = ax, kind = 'barh')
plt.rcParams['figure.figsize'] = [10, 25]
plt.grid()
ax.set_xlim([0, 100])
ax.tick_params(top=True, labeltop=True, bottom=True, labelbottom=True)
plt.show()

In [ ]:
def pie_charts_by_category(df: pd.DataFrame, label: str):
    # Compute the maximum percentage for each category to sort by
    category_max_percentage = (
        df['%']
        .groupby(level=0)  # Group by category
        .max()  # Get the maximum percentage for each category
    )

    # Sort categories by the highest top percentage (descending)
    sorted_categories = category_max_percentage.sort_values(ascending=False).index

    # Loop through each category in the sorted order
    for category in sorted_categories:
        # Extract the data for the current category
        group = df.loc[category]
        counts = group['Count']
        percentages = group['%']
        
        # Filter labels: Only show for rows with percentage > 20%
        labels = [
            label if percentage > 5 else '' 
            for label, percentage in zip(counts.index, percentages)
        ]
        
        # Define a function to display percentages > 20% on the pie chart
        def autopct_func(pct):
            return f'{pct:.1f}%' if pct > 5 else ''

        # Create the pie chart with smaller figure size
        plt.figure(figsize=(4, 4))  # Reduced size
        plt.pie(
            counts, 
            labels=labels,  # Use filtered labels
            autopct=autopct_func,  # Use custom function for autopct
            startangle=90
        )
        
        # Set title and show the chart
        plt.title(f'{category} ({label}; n={counts.sum()})')
        plt.axis('equal')  # Ensures the pie chart is circular
        plt.show()

In [ ]:
pie_charts_by_category(mx_freq_df, 'mail')

In [ ]:
fig, ax = plt.subplots()
webhost_freq_df["%"].sort_values().plot(ax = ax, kind = 'barh', ylabel='frequency')
plt.grid()
ax.set_xlim([0, 100])
ax.tick_params(top=True, labeltop=True, bottom=True, labelbottom=True)
plt.show()

In [ ]:
pie_charts_by_category(webhost_freq_df, "webserver")